<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/03_recsys_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Théorie des systèmes de recommandation

## Bloc 1 : popularité, filtrage collaboratif user-based et item-based

### Le cadre : la matrice d'interactions R
Un système de recommandation travaille sur une matrice **R** (lignes = "utilisateurs", colonnes = articles).
R[u, i] = 1 si l'utilisateur u a interagi avec l'article i, 0 sinon.
Objectif : pour un utilisateur u, ordonner les articles non vus et garder les 20 meilleurs.

**Dans OTTO** : une ligne = une *session* (pas d'identité persistante).

In [1]:
n_sessions = 12_899_779
n_items    = 1_855_603
mean_distinct = 10.36327                      # mesuré au notebook 2

n_pairs = mean_distinct * n_sessions          # nombre de cases non nulles dans R
n_cells = n_sessions * n_items

print(f"Cases non nulles : {n_pairs:,.0f}")
print(f"Cases totales    : {n_cells:.3e}")
print(f"Densité          : {n_pairs / n_cells * 100:.5f} %")
print(f"R dense en 1 octet/case : {n_cells / 1e12:.1f} To")

Cases non nulles : 133,683,893
Cases totales    : 2.394e+13
Densité          : 0.00056 %
R dense en 1 octet/case : 23.9 To


**les trois méthodes sur une mini-matrice**

In [2]:
import numpy as np
import pandas as pd

items = ["A", "B", "C", "D", "E", "F"]
R = pd.DataFrame(
    [[1,1,0,0,0,0],   # s0
     [1,1,1,0,0,0],   # s1
     [0,1,1,1,0,0],   # s2
     [0,0,0,1,1,1],   # s3
     [1,0,1,0,0,0]],  # s4
    columns=items, index=[f"s{i}" for i in range(5)])
Rm = R.values.astype(float)
display(R)

def recommend(scores, seen, k=3):
    s = pd.Series(scores, index=items).drop(seen)      # on retire les articles déjà vus
    return s.sort_values(ascending=False).head(k)

def run(query_items):
    q = np.array([1.0 if it in query_items else 0.0 for it in items])

    # 1) popularité
    pop = Rm.sum(axis=0)

    # 2) user-based : cosinus entre la requête et chaque session
    sim_users = (Rm @ q) / (np.linalg.norm(Rm, axis=1) * np.linalg.norm(q))
    user_scores = sim_users @ Rm

    # 3) item-based : cosinus entre colonnes
    col_norm = np.linalg.norm(Rm, axis=0)
    S = (Rm.T @ Rm) / np.outer(col_norm, col_norm)
    np.fill_diagonal(S, 0)                              # un article n'est pas son propre voisin
    item_scores = q @ S

    print(f"\n=== Session qui a vu {query_items} ===")
    print("Popularité :\n", recommend(pop, query_items).round(3))
    print("User-based :\n", recommend(user_scores, query_items).round(3))
    print("Item-based :\n", recommend(item_scores, query_items).round(3))

run(["A"])
run(["E"])

,A,B,C,D,E,F
s0,1,1,0,0,0,0
s1,1,1,1,0,0,0
s2,0,1,1,1,0,0
s3,0,0,0,1,1,1
s4,1,0,1,0,0,0



=== Session qui a vu ['A'] ===
Popularité :
 B    3.0
C    3.0
D    2.0
dtype: float64
User-based :
 B    1.284
C    1.284
D    0.000
dtype: float64
Item-based :
 B    0.667
C    0.667
D    0.000
dtype: float64

=== Session qui a vu ['E'] ===
Popularité :
 A    3.0
B    3.0
C    3.0
dtype: float64
User-based :
 F    0.577
D    0.577
A    0.000
dtype: float64
Item-based :
 F    1.000
D    0.707
A    0.000
dtype: float64


## **bloc 2 : factorisation matricielle et content-based**

### 4. Factorisation matricielle (Matrix Factorization)

**Idée.** Au lieu de comparer des vecteurs de 1,86 M de dimensions, on résume chaque session et chaque article par un petit vecteur de **k facteurs latents** (k ≈ 32 à 128) :

    R ≈ P · Qᵀ        P : sessions × k     Q : articles × k
    score(u, i) = p_u · q_i

On cherche P et Q qui minimisent l'erreur de reconstruction :

    min Σ (R[u,i] − p_u · q_i)² + λ (‖p_u‖² + ‖q_i‖²)

λ est la régularisation (empêche le surapprentissage).

**Trois variantes à connaître :**
- **SVD tronquée** : solution exacte, optimale en erreur quadratique, traite les cases à 0 comme de vraies valeurs.
- **ALS / implicit feedback** (Hu, Koren, Volinsky) : les 0 signifient « pas observé », pas « pas aimé ». Chaque case a une *confiance* c = 1 + α·(nombre d'interactions).
- **BPR** : on n'optimise plus l'erreur, mais l'ordre : un article vu doit passer avant un article non vu.

**Intuition.** Deux articles souvent vus dans les mêmes sessions obtiennent des vecteurs proches, même s'ils ne co-occurrent jamais directement : la factorisation généralise.

# **content-based**

### 5. Content-Based Filtering

**Idée.** Recommander des articles *ressemblant* à ceux déjà vus, d'après leurs **attributs** (catégorie, marque, prix, texte, image).

    sim(i, j) = cos(features_i, features_j)

| Avantages | Limites |
|---|---|
| Aucun cold start d'article | Exige des attributs de qualité |
| Explicable | Pas de surprise (« bulle de filtre ») |
| Ne dépend pas des autres utilisateurs | Ne capte pas les complémentarités (on ne commande pas deux téléphones) |

**Dans OTTO : ce n'est pas applicable directement.** Les données ne contiennent que `aid`, `ts` et `type` : aucun attribut d'article. L'`aid` est un identifiant anonymisé, sans sens.
On peut seulement construire des *représentations apprises* à partir des sessions (c'est le rôle de la factorisation et des embeddings de bloc 3).

**Remarque.** L'EDA a montré que 100 % des `aid` du test existent dans le train, donc le principal avantage du content-based (cold start) a peu de valeur ici.

# **Lien avec OTTO**

### Factorisation sur OTTO : ordres de grandeur
- Matrice R : 12,9 M × 1,86 M, 133,7 M cases non nulles (densité 0,00056 %).
- Avec k = 64 en float32 :
    - facteurs articles Q : 1,86 M × 64 × 4 octets ≈ **0,48 GB**
    - facteurs sessions P : 12,9 M × 64 × 4 octets ≈ **3,3 GB**
- **Problème des sessions de test** : elles ne sont pas dans P. Il faut calculer leur vecteur à la volée (*fold-in*), par exemple en moyennant les q_i de leurs articles. Sur des sessions de 2 événements, ce vecteur est très pauvre.
- **Coût** : ALS tourne en O(nnz·k²) par itération, faisable, mais on perd les informations d'ordre, de type d'événement et de temps, que la matrice R binaire n'encode pas.

**SVD sue la mini matrice**

In [3]:
import numpy as np, pandas as pd

items = list("ABCDEF")
R = np.array([[1,1,0,0,0,0],
              [1,1,1,0,0,0],
              [0,1,1,1,0,0],
              [0,0,0,1,1,1],
              [1,0,1,0,0,0]], dtype=float)

U, s, Vt = np.linalg.svd(R, full_matrices=False)
print("Valeurs singulières :", s.round(3))

# Erreur de reconstruction selon le nombre de facteurs
for k in (1, 2, 3):
    Rk = (U[:, :k] * s[:k]) @ Vt[:k]
    print(f"k={k} : erreur de reconstruction = {np.linalg.norm(R - Rk):.3f}")

# Vecteurs latents des articles (k=2)
k = 2
V = Vt[:k].T
display(pd.DataFrame(V, index=items, columns=["facteur 1", "facteur 2"]).round(3))

# Recommandation pour une session inconnue (fold-in par projection)
def recommend_svd(seen, topn=3):
    q = np.array([1.0 if it in seen else 0.0 for it in items])
    scores = q @ V @ V.T                       # projection dans l'espace latent
    out = pd.Series(scores, index=items).drop(seen).sort_values(ascending=False)
    print(f"\nSession qui a vu {seen} :")
    print(out.head(topn).round(3))

recommend_svd(["A"])
recommend_svd(["E"])

Valeurs singulières : [2.698 1.836 1.134 1.    0.252]
k=1 : erreur de reconstruction = 2.392
k=2 : erreur de reconstruction = 1.533
k=3 : erreur de reconstruction = 1.031


,facteur 1,facteur 2
A,0.535,0.308
B,0.572,0.029
C,0.572,0.029
D,0.234,-0.662
E,0.044,-0.483
F,0.044,-0.483



Session qui a vu ['A'] :
B    0.315
C    0.315
D   -0.079
dtype: float64

Session qui a vu ['E'] :
D    0.330
F    0.235
C    0.012
dtype: float64


# **bloc 3 : session-based, co-visitation et séquentiel**

### 6. Recommandation basée sur la session (Session-Based)

**Principe.** On ne connaît pas l'utilisateur, seulement la **session en cours** : la liste (ordonnée) de ses événements. On prédit la suite à partir d'elle seule.

**Pourquoi c'est le bon cadre pour OTTO (mesures du notebook 2) :**
- pas d'identité persistante : une ligne = une session ;
- sessions de test très courtes (médiane 2 événements), donc on ne peut pas apprendre un profil d'utilisateur ;
- 95,6 % des orders (sessions complètes) portent sur un article déjà vu : la session contient l'essentiel du signal ;
- le test suit immédiatement le train : les relations entre articles apprises sur le train restent utiles.

**Deux sources de candidats :**
1. les articles de la session elle-même (re-recommandation) ;
2. les articles reliés aux siens par une table de similarité (co-visitation).

# **7. Co-visitation**

**Idée.** Item-based simplifié : on **compte** combien de fois deux articles apparaissent dans la même session, sans normaliser.

    C[i, j] = Σ_sessions  poids(i, j)        poids = 1 pour chaque paire (i, j) de la session

**Recommandation** pour une session S : score(j) = Σ_{i ∈ S} C[i, j], puis on garde les 20 meilleurs j ∉ S.

**Trois choix de conception (chacun est une hypothèse à tester en Phase 6) :**

| Choix | Variantes | Pourquoi ça compte dans OTTO |
|---|---|---|
| **Fenêtre temporelle** | toute la session / 24 h / les K derniers événements | les sessions durent plusieurs jours (55 % > 24 h) : relier deux articles vus à 17 h d'écart est du bruit |
| **Poids de la paire** | 1 / selon le type de l'événement cible / décroissance avec l'écart de temps | un order est un signal plus fort qu'un click |
| **Direction** | symétrique / orientée (i avant j) | « j après i » prédit le futur, ce qui est exactement la tâche |

**Séparer les matrices par cible.** Une table pour clicks, une pour carts, une pour orders : H1 a montré que les tops diffèrent selon le type (7 articles sur 20 en commun entre clicks et orders).

**Le problème : l'explosion des paires.** Une session de n événements produit ~n² paires. Une session à 500 événements en produit 250 000. Solutions :
- ne garder que les **K derniers** événements de chaque session ;
- ne garder, pour chaque article i, que ses **N meilleurs voisins** j ;
- traiter par blocs et sauvegarder des résultats intermédiaires.

**Limite.** La co-visitation brute favorise les articles populaires (ils co-occurrent avec tout). On peut normaliser (par la popularité de j, ou cosinus), mais c'est à mesurer.

# **co-visitation sur une mini-donnée**

In [4]:
from collections import defaultdict, Counter

# (aid, heure, type) ; l'heure est en heures depuis le début
sessions = [
    [("A",0,"clicks"), ("B",1,"clicks"), ("C",2,"carts"), ("C",3,"orders")],
    [("A",0,"clicks"), ("B",1,"clicks"), ("D",2,"clicks")],
    [("B",0,"clicks"), ("C",1,"clicks"), ("C",2,"carts")],
    [("A",0,"clicks"), ("E",40,"clicks"), ("E",41,"carts")],    # E vu 40 h plus tard
    [("D",0,"clicks"), ("B",1,"clicks"), ("C",2,"carts")],
]
WINDOW = 24
TW = {"clicks": 1, "carts": 3, "orders": 6}      # poids par type de l'événement cible

def covisit(sessions, window=None, type_w=False, directed=False):
    M = defaultdict(Counter)
    for s in sessions:
        for a in range(len(s)):
            for b in range(len(s)):
                if a == b: continue
                ia, ta, _  = s[a]
                ib, tb, yb = s[b]
                if ia == ib: continue
                if directed and tb <= ta: continue        # b doit venir après a
                if window is not None and abs(tb - ta) > window: continue
                M[ia][ib] += TW[yb] if type_w else 1
    return M

def rec(M, seen, k=3):
    sc = Counter()
    for it in seen:
        for j, w in M[it].items():
            if j not in seen:
                sc[j] += w
    return sc.most_common(k)

for name, kw in [("sans fenêtre", {}),
                 ("fenêtre 24 h", dict(window=WINDOW)),
                 ("fenêtre 24 h + poids par type", dict(window=WINDOW, type_w=True))]:
    M = covisit(sessions, **kw)
    print(f"{name:<32} | A -> {rec(M, ['A'])} | B -> {rec(M, ['B'])}")

sans fenêtre                     | A -> [('B', 2), ('C', 2), ('E', 2)] | B -> [('C', 5), ('A', 2), ('D', 2)]
fenêtre 24 h                     | A -> [('B', 2), ('C', 2), ('D', 1)] | B -> [('C', 5), ('A', 2), ('D', 2)]
fenêtre 24 h + poids par type    | A -> [('C', 9), ('B', 2), ('D', 1)] | B -> [('C', 16), ('A', 2), ('D', 2)]


### 8. Recommandation séquentielle (Sequential)

**Idée.** L'ordre compte : on modélise P(prochain article | historique ordonné).

| Modèle | Principe | Remarque |
|---|---|---|
| **Chaîne de Markov** | P(j \| dernier article i) estimée par comptage des transitions i → j | simple, c'est de la co-visitation orientée |
| **Item2Vec** (word2vec) | les articles d'une session sont des « mots » d'une « phrase » ; on apprend des embeddings | donne des similarités plus lisses que le comptage |
| **GRU4Rec** (RNN) | un GRU lit la session événement par événement | séquences courtes OK, mais entraînement lent |
| **SASRec / Transformers** | attention sur l'ensemble de la séquence | puissant, coûteux, gourmand en données |

**Pour OTTO, avec réserve (hypothèses, pas résultats) :**
- La session de test médiane n'a que 2 événements : il y a peu de séquence à exploiter, donc un gros avantage des modèles séquentiels sur la co-visitation orientée n'est pas garanti.
- Ils pourraient aider sur les sessions longues (5,7 % du test dure plus de 24 h, et des sessions atteignent des centaines d'événements).
- Règle de ton cahier des charges : ne pas utiliser le deep learning parce qu'il est plus complexe, mais parce qu'une expérience le justifie.